# Milestone 2 — Worked Example: Representation & Language-Model Comparison (FOMC corpus)
**ISBA 2411 · Reference / launchpad for Milestone 2**

A guided worked example on the **provided FOMC transcript corpus**, contrasting **classical (NLTK)**
and **modern (Hugging Face)** approaches. Use it to see the techniques end-to-end, then adapt the
same *baseline → measure* structure to your team's own dataset in
`Milestone2_Baseline_and_Representation.ipynb`.

**Reading connections**
| Concept | Where to read |
|---|---|
| N-gram language models & perplexity | J&M Ch. 3 |
| Classical vs. subword tokenization | J&M Ch. 2; HOLLM Ch. 2 |
| Neural generation & decoding | HOLLM Ch. 7; Tunstall Ch. 5 |
| Embeddings / representation models | HOLLM Ch. 10 |

Written analysis: **200–300 words per section.**

In [ ]:
# Setup (Colab)
!pip install -q nltk transformers torch
import nltk, math, torch
nltk.download("punkt")

## Load the FOMC corpus
The corpus ships in the repo at `data/fomc_corpus.txt` (FOMC verbatim meeting transcripts for 2019 — 8 meetings, ~3.3 MB, ~29k sentences). It is built by `data/fetch_fomc.py`, which downloads the transcript PDFs from the Federal Reserve and extracts the text. Re-run that script to refresh or choose a different year.

In [ ]:
# Load the provided FOMC corpus (data/fomc_corpus.txt -- 2019 meetings, 8 transcripts).
# On Colab, clone the course repo first, e.g.:
#   !git clone https://github.com/JulesMalin/isba2411-nlp.git && cd isba2411-nlp
import os

candidates = ["data/fomc_corpus.txt", "../data/fomc_corpus.txt",
              "isba2411-nlp/data/fomc_corpus.txt"]
path = next((c for c in candidates if os.path.exists(c)), None)
if path is None:
    raise FileNotFoundError(
        "fomc_corpus.txt not found. Run `python data/fetch_fomc.py` from the "
        "repo root to build it, or adjust the path above.")

with open(path, encoding="utf-8") as f:
    text = f.read()

sentences = nltk.sent_tokenize(text)
print(f"loaded {len(text):,} chars  |  {len(sentences):,} sentences  from {path}")

## Part A — Classical n-gram (NLTK) vs. neural (GPT-2) language models

### A1. NLTK trigram model with Laplace smoothing
Fit a trigram model and compute perplexity on a held-out slice.

In [ ]:
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import Laplace
# tokens = [nltk.word_tokenize(s.lower()) for s in sentences]
# train, vocab = padded_everygram_pipeline(3, tokens)
# lm = Laplace(3); lm.fit(train, vocab)
# TODO: compute perplexity on a held-out set with lm.perplexity(...)

### A2. Modern Interlude — classical vs. modern tokenization
The single biggest shift from classical to modern NLP is the move from **rule-based word tokens** to
**learned subword (BPE) tokens**. Compare them on the same FOMC sentence.

| Classical (NLTK / J&M) | Modern (Hugging Face / HOLLM & Tunstall) |
|---|---|
| `word_tokenize` — whitespace/rules | `AutoTokenizer` BPE — learned from data |
| Laplace trigram perplexity | GPT-2 cross-entropy perplexity |
| N-gram generation | Transformer generation |

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("gpt2")
sample = "The Committee judges that inflation has moderated somewhat."
print("NLTK word tokens:", nltk.word_tokenize(sample))
print("GPT-2 BPE tokens:", tok.tokenize(sample))
print("counts -> words:", len(nltk.word_tokenize(sample)), "| subwords:", len(tok.tokenize(sample)))

### A3. GPT-2 perplexity (neural baseline)
Compute GPT-2 perplexity on the same held-out text via cross-entropy loss.

In [ ]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("gpt2"); model.eval()
# enc = tok(held_out_text, return_tensors="pt")
# with torch.no_grad():
#     loss = model(**enc, labels=enc["input_ids"]).loss
# print("GPT-2 perplexity:", math.exp(loss.item()))

### A — Reflection (200–300 words)
Compare the trigram and GPT-2 perplexities. Where does the n-gram model fail (rare words, long
dependencies, OOV)? How does subword tokenization change what "a token" means for perplexity?

## Part B — Decoding strategies
Generate FOMC-style text with greedy, beam, and nucleus sampling; compare quality.

In [ ]:
prompt = "The Committee decided to"
ids = tok(prompt, return_tensors="pt").input_ids
greedy  = model.generate(ids, max_new_tokens=40, do_sample=False)
beam    = model.generate(ids, max_new_tokens=40, num_beams=5, no_repeat_ngram_size=2)
nucleus = model.generate(ids, max_new_tokens=40, do_sample=True, top_p=0.9, temperature=0.8)
for name,out in [("greedy",greedy),("beam",beam),("nucleus",nucleus)]:
    print(f"\n[{name}]\n", tok.decode(out[0], skip_special_tokens=True))

### B — Reflection (200–300 words)
Which decoding strategy reads most like real FOMC language, and why? Discuss the
coherence-vs-diversity tradeoff (greedy/beam repetition vs. nucleus variability).

## Adapting this to your Milestone 2
Your graded Milestone 2 runs the same *represent → baseline → measure* loop on **your team's own
dataset** (see `Milestone2_Baseline_and_Representation.ipynb`). This FOMC example is a reference for
the tokenization/perplexity/decoding mechanics — your project's task and metric may differ.